# Full Demo Integration Test

Run the production state machine without parameter editing controls. Edit JSON on disk, reload config, then test one step or the complete mission.

The reusable HUD shows mission/subphase timing, effective base commands, perception health, retries and recent transitions. Each AVI recording is accompanied by a synchronized .events.csv file with one telemetry row per video frame. With `static_detection` enabled, **Start Media** loads the production can/Tag detectors and draws their boxes even while the FSM is idle; choose Can, Tag, or both with `static_target`. Static inference stops automatically whenever the full demo worker is running, so detector objects are never called concurrently with the FSM.

In [ ]:
from __future__ import print_function

import contextlib
import io
import os
import sys
import threading
import time
import traceback

def find_project_root():
    current = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isfile(os.path.join(current, 'config.json')):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError('config.json not found')

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import cv2
import ipywidgets as widgets
from IPython.display import FileLink, Javascript, display

from demo_core import DemoDiagnostics, DemoStateMachine, HudEventRecorder, load_config, render_hud, runtime_hud_snapshot
from demo_core.config import DIAGNOSTIC_OUTPUT_DIR
from demo_core.logging_utils import default_log_path

disk_config = load_config()
runtime = None
demo_thread = None
demo_thread_runtime = None
media_thread = None
media_running = False
last_recording_path = None
last_event_path = None
output = widgets.Output(layout={'border': '1px solid #ccc', 'height': '560px', 'overflow_y': 'auto'})
image = widgets.Image(format='jpeg', width=320, height=240)
reload_models = widgets.Checkbox(value=True, description='reload_models')
compact_log = widgets.Checkbox(value=True, description='compact_log')
camera_real = widgets.Checkbox(value=True, description='camera_real')
base_real = widgets.Checkbox(value=True, description='base_real')
arm_real = widgets.Checkbox(value=True, description='arm_real')
exp2 = widgets.Checkbox(value=True, description='exp2')
live_stream = widgets.Checkbox(value=True, description='live_stream')
record_camera = widgets.Checkbox(value=False, description='record_camera')
hud_enabled = widgets.Checkbox(value=True, description='hud')
static_detection = widgets.Checkbox(value=True, description='static_detection')
routine2 = widgets.Checkbox(value=False, description='routine2_square')
static_target = widgets.Dropdown(options=[('Can + Tag', 'both'), ('Can', 'can'), ('Tag', 'bin')], value='both', description='static_target')
detection_fps = widgets.FloatText(value=2.0, description='detection_fps')
media_fps = widgets.FloatText(value=3.0, description='media_fps')
preview_fps = widgets.FloatText(value=1.0, description='preview_fps')

class Tee(object):
    def __init__(self, *streams): self.streams = streams
    def write(self, data):
        for stream in self.streams: stream.write(data); stream.flush()
    def flush(self):
        for stream in self.streams: stream.flush()

class WidgetStream(object):
    def __init__(self, widget, stderr=False):
        self.widget = widget
        self.stderr = bool(stderr)
    def write(self, data):
        if data:
            append = self.widget.append_stderr if self.stderr else self.widget.append_stdout
            append(str(data))
    def flush(self):
        return None

class ImportantStream(object):
    def __init__(self, stream):
        self.stream = stream
        self.pending = ''
    def write(self, data):
        self.pending += data
        while '\n' in self.pending:
            line, self.pending = self.pending.split('\n', 1)
            if important(line): self.stream.write(line + '\n')
    def flush(self):
        if self.pending:
            if important(self.pending): self.stream.write(self.pending)
            self.pending = ''
        self.stream.flush()

class ThreadOutputRouter(object):
    def __init__(self, fallback):
        self.fallback = fallback
        self.routes = {}
        self.lock = threading.Lock()
    def bind(self, stream):
        with self.lock:
            self.routes.setdefault(threading.get_ident(), []).append(stream)
    def unbind(self):
        with self.lock:
            stack = self.routes.get(threading.get_ident(), [])
            if stack: stack.pop()
            if not stack: self.routes.pop(threading.get_ident(), None)
    def _target(self):
        with self.lock:
            stack = self.routes.get(threading.get_ident(), [])
            return stack[-1] if stack else self.fallback
    def write(self, data):
        return self._target().write(data)
    def flush(self):
        return self._target().flush()
    def isatty(self):
        return False
    def __getattr__(self, name):
        return getattr(self.fallback, name)

def _base_stream(stream):
    seen = set()
    while hasattr(stream, 'fallback') and id(stream) not in seen:
        seen.add(id(stream))
        stream = stream.fallback
    return stream

stdout_router = ThreadOutputRouter(_base_stream(sys.stdout))
stderr_router = ThreadOutputRouter(_base_stream(sys.stderr))
sys.stdout = stdout_router
sys.stderr = stderr_router

@contextlib.contextmanager
def routed_output(stdout_stream, stderr_stream=None):
    stdout_router.bind(stdout_stream)
    stderr_router.bind(stderr_stream or stdout_stream)
    try:
        yield
    finally:
        stderr_router.unbind()
        stdout_router.unbind()

def scroll_log():
    display(Javascript("setTimeout(function(){var x=document.querySelectorAll('.widget-output');for(var i=0;i<x.length;i++){var e=x[i];if(!e.__demoAutoScroll){var o=new MutationObserver(function(m){var t=m[0].target.closest('.widget-output');if(t){t.scrollTop=t.scrollHeight;}});o.observe(e,{childList:true,subtree:true,characterData:true});e.__demoAutoScroll=o;}e.scrollTop=e.scrollHeight;}},80);"))

def important(line):
    tokens = ('[fsm]', '[mission]', '[exp2] phase=', '[map] bin tag localization', '[error]', 'Traceback', 'FAILED', 'DONE', '[integration]', '[result]')
    return any(token in line for token in tokens)

def logged(fn):
    def wrapped(_=None):
        with output:
            path = default_log_path('integration_{}'.format(fn.__name__))
            with open(path, 'a') as log_file:
                widget_stream = WidgetStream(output)
                display_stream = ImportantStream(widget_stream) if compact_log.value else widget_stream
                stdout_tee = Tee(log_file, display_stream)
                stderr_tee = Tee(log_file, WidgetStream(output, stderr=True))
                with routed_output(stdout_tee, stderr_tee):
                    start = time.time(); status = 'SUCCESS'
                    try:
                        print('\n>>> {} log={}'.format(fn.__name__, path))
                        result = fn()
                        print('[result] {}'.format(result))
                    except Exception as exc:
                        status = 'FAILED'; print('[error] {}'.format(exc)); traceback.print_exc()
                    print('[integration] {} elapsed={:.1f}s'.format(status, time.time() - start))
            scroll_log()
    return wrapped

def immediate(fn):
    def wrapped(_=None):
        with output:
            try:
                print('\n>>> {}'.format(fn.__name__))
                print('[result] {}'.format(fn()))
            except Exception as exc:
                print('[error] {}'.format(exc))
                traceback.print_exc()
        scroll_log()
    return wrapped

def overrides():
    use_square = bool(routine2.value)
    routine_id = 2 if use_square else 0
    bin_x_m, bin_y_m = (0.05, -7) if use_square else (-0.10, -0.6)
    return {
        'runtime': {'dry_run': {'camera': not camera_real.value, 'base': not base_real.value, 'arm': not arm_real.value}},
        'navigation': {
            'can': {'search': {'routine': routine_id}},
            'bin': {'search': {'routine': routine_id}, 'side_docking': {'experimental': {'enabled': bool(exp2.value)}}},
        },
        'vague_map': {
            'bin_marker_position': {'x_m': bin_x_m, 'y_m': bin_y_m},
            'bin_docking_pose': {'x_m': bin_x_m, 'y_m': bin_y_m},
            'bin_side_docking_pose': {'x_m': bin_x_m, 'y_m': bin_y_m},
        },
    }

def reload_config():
    global disk_config, runtime
    if runtime is not None:
        runtime.release_camera()
    disk_config = load_config(overrides=overrides())
    runtime = DemoStateMachine(disk_config)
    return {'config': disk_config.parameters_path, 'dry_run': disk_config.get('runtime.dry_run'), 'search_routine': disk_config.get('navigation.can.search.routine'), 'bin_xy_m': (disk_config.get('vague_map.bin_marker_position.x_m'), disk_config.get('vague_map.bin_marker_position.y_m')), 'exp2': disk_config.get('navigation.bin.side_docking.experimental')}

def current():
    global runtime
    if runtime is None:
        reload_config()
    return runtime

def show_state():
    rt = current()
    return {'state': rt.state.value, 'context': rt.context.snapshot()}

def start_camera():
    current().start_camera()
    return show_state()

def load_models():
    current().load_detectors(bool(reload_models.value))
    return show_state()

def preflight():
    diag = DemoDiagnostics(current().config, services=current().services)
    return diag.preflight(bool(reload_models.value))

def step_once():
    outcome = current().step_once()
    return {'state': current().state.value, 'event': outcome.event.value if outcome.event else None, 'reason': outcome.reason}

def _demo_running():
    return demo_thread_runtime is not None or (demo_thread is not None and demo_thread.is_alive())

def _static_observations(rt, frame):
    observations = []
    mode = static_target.value
    with routed_output(io.StringIO()):
        if mode in ('can', 'both'):
            observation = rt.services.can_detector.detect(frame)
            observation = dict(observation or {})
            observation['kind'] = 'can'
            observations.append(observation)
        if mode in ('bin', 'both'):
            observation = rt.services.bin_detector.detect(frame)
            observation = dict(observation or {})
            observation['kind'] = 'bin'
            observations.append(observation)
    return observations

def _static_snapshot(rt, depth_stats, frame, recording, observations):
    snapshot = runtime_hud_snapshot(rt, depth_stats, frame_available=frame is not None, recording=recording)
    found = [observation for observation in observations if observation.get('found')]
    primary = max(found, key=lambda value: float(value.get('confidence', 0.0))) if found else None
    snapshot['subphase'] = 'static_detect_{}'.format(static_target.value)
    snapshot['target'] = static_target.value
    snapshot['target_found'] = bool(primary)
    snapshot['bbox'] = None
    snapshot['center_x'] = None
    snapshot['center_y'] = None
    if primary is not None:
        pose = primary.get('pose') or {}
        snapshot.update({
            'confidence': primary.get('confidence'), 'error_x': primary.get('error_x'),
            'bbox_height_norm': primary.get('bbox_height_norm'), 'target_distance': primary.get('distance'),
            'pose_x': pose.get('x'), 'pose_y': pose.get('y'), 'pose_z': pose.get('z'),
        })
    else:
        snapshot.update({'confidence': None, 'error_x': None, 'bbox_height_norm': None, 'target_distance': None, 'pose_x': None, 'pose_y': None, 'pose_z': None})
    return snapshot

def _draw_static_boxes(canvas, observations, frame_shape):
    source_height, source_width = frame_shape[:2]
    scale_x = canvas.shape[1] / float(max(1, source_width)); scale_y = canvas.shape[0] / float(max(1, source_height))
    colors = {'can': (0, 255, 0), 'bin': (255, 0, 255)}
    for observation in observations:
        box = observation.get('bbox') if observation.get('found') else None
        if not box or len(box) != 4:
            continue
        x1, y1, x2, y2 = box; kind = observation.get('kind', 'target'); color = colors.get(kind, (0, 255, 255))
        p1 = (int(float(x1) * scale_x), int(float(y1) * scale_y)); p2 = (int(float(x2) * scale_x), int(float(y2) * scale_y))
        cv2.rectangle(canvas, p1, p2, color, 2)
        cv2.putText(canvas, '{} {:.2f}'.format(kind.upper(), float(observation.get('confidence', 0.0))), (p1[0], max(16, p1[1] - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)
    return canvas

def _render_media_frame(rt, frame, depth_stats, recording, static_observations=None):
    observations = static_observations
    if observations is None:
        last = getattr(rt.context, 'last_observation', None)
        observations = [last] if last else []
    use_rectified = any(item and item.get('frame_space') == 'rectified' for item in observations)
    if use_rectified:
        frame = rt.services.can_detector.display_frame(frame)
        if static_observations is not None:
            static_observations = [rt.services.can_detector.observation_in_display_space(item) for item in static_observations]
    if static_observations is not None:
        snapshot = _static_snapshot(rt, depth_stats, frame, recording, static_observations)
    else:
        snapshot = runtime_hud_snapshot(rt, depth_stats, frame_available=frame is not None, recording=recording)
    rendered = render_hud(frame, snapshot, rt.config.section('camera'), enabled=bool(hud_enabled.value))
    if static_observations is not None and hud_enabled.value:
        _draw_static_boxes(rendered, static_observations, frame.shape)
    return rendered, snapshot

def _media_loop(rt):
    global media_running, media_thread, last_recording_path, last_event_path
    writer = None
    event_recorder = None
    writer_path = None
    recording_started_at = None
    depth_stats = None
    last_depth_sample = 0.0
    last_frame_signature = None
    stale_since = time.time()
    stale_reported = False
    last_preview_at = 0.0
    last_record_at = 0.0
    last_detection_at = 0.0
    static_observations = []
    try:
        while media_running:
            frame = rt.services.depth.read_frame()
            if frame is not None:
                now = time.time()
                signature = hash(frame[::16, ::16].tobytes())
                if signature != last_frame_signature:
                    last_frame_signature = signature
                    stale_since = now
                    stale_reported = False
                elif not stale_reported and now - stale_since >= 2.0:
                    output.append_stderr('[media] warning: camera frame unchanged for 2 seconds; detector may also be seeing a stale frame\n')
                    stale_reported = True
                if hud_enabled.value and now - last_depth_sample >= 1.0:
                    cached = rt.services.depth.latest_lens_stats
                    depth_stats = dict(cached) if cached else None
                    last_depth_sample = now
                record_interval = 1.0 / max(1.0, float(media_fps.value))
                preview_interval = 1.0 / max(0.2, float(preview_fps.value))
                record_due = bool(record_camera.value) and now - last_record_at >= record_interval
                preview_due = bool(live_stream.value) and now - last_preview_at >= preview_interval
                static_mode = bool(static_detection.value) and not _demo_running()
                if static_mode and now - last_detection_at >= 1.0 / max(0.2, float(detection_fps.value)):
                    static_observations = _static_observations(rt, frame)
                    last_detection_at = now
                if not static_mode:
                    static_observations = None
                rendered = None; render_snapshot = None
                if record_due or preview_due:
                    rendered, render_snapshot = _render_media_frame(rt, frame, depth_stats, record_camera.value, static_observations)
                if preview_due and rendered is not None:
                    preview = cv2.resize(rendered, (320, 240), interpolation=cv2.INTER_AREA)
                    ok, encoded = cv2.imencode('.jpg', preview, [int(cv2.IMWRITE_JPEG_QUALITY), 55])
                    if ok: image.value = encoded.tobytes()
                    last_preview_at = now
                if record_camera.value and writer is None:
                    if not os.path.isdir(str(DIAGNOSTIC_OUTPUT_DIR)):
                        os.makedirs(str(DIAGNOSTIC_OUTPUT_DIR))
                    writer_path = os.path.join(str(DIAGNOSTIC_OUTPUT_DIR), 'full_demo_{}.avi'.format(int(time.time() * 1000)))
                    height, width = 480, 640
                    fps = max(1.0, float(media_fps.value))
                    writer = cv2.VideoWriter(writer_path, cv2.VideoWriter_fourcc(*'MJPG'), fps, (width, height))
                    if not writer.isOpened():
                        writer.release(); writer = None
                        raise RuntimeError('camera recorder could not open {}'.format(writer_path))
                    last_recording_path = writer_path
                    last_event_path = os.path.splitext(writer_path)[0] + '.events.csv'
                    event_recorder = HudEventRecorder(last_event_path)
                    recording_started_at = now
                    output.append_stdout('[media] recording started {} events={}\n'.format(writer_path, last_event_path))
                if writer is not None:
                    if record_camera.value:
                        if record_due and rendered is not None:
                            writer.write(rendered)
                            event_recorder.write(render_snapshot, now - recording_started_at)
                            last_record_at = now
                    else:
                        writer.release(); writer = None
                        event_recorder.close(); event_recorder = None
                        output.append_stdout('[media] recording stopped {} events={}\n'.format(writer_path, last_event_path))
            active_fps = max(float(media_fps.value) if record_camera.value else 0.0, float(preview_fps.value) if live_stream.value else 0.0, 0.2)
            time.sleep(1.0 / active_fps)
    except Exception as exc:
        output.append_stderr('[media] error: {}\n'.format(exc))
        output.append_stderr(traceback.format_exc())
    finally:
        if writer is not None:
            writer.release()
            output.append_stdout('[media] recording stopped {}\n'.format(writer_path))
        if event_recorder is not None:
            event_recorder.close()
        media_running = False
        media_thread = None
        output.append_stdout('[media] monitor stopped\n')

def start_media(start_camera=True):
    global media_running, media_thread
    if not live_stream.value and not record_camera.value:
        return {'started': False, 'reason': 'enable live_stream or record_camera'}
    if media_thread is not None and media_thread.is_alive():
        return {'started': False, 'reason': 'media monitor already running'}
    rt = current()
    if start_camera:
        rt.start_camera()
        if static_detection.value:
            rt.load_detectors(bool(reload_models.value))
    media_running = True
    media_thread = threading.Thread(target=_media_loop, args=(rt,))
    media_thread.daemon = True
    media_thread.start()
    return {'started': True, 'live': bool(live_stream.value), 'recording': bool(record_camera.value), 'static_detection': bool(static_detection.value), 'static_target': static_target.value}

def stop_media(wait_seconds=5.0):
    global media_running, media_thread
    media_running = False
    thread = media_thread
    if thread is not None and thread is not threading.current_thread() and float(wait_seconds) > 0.0:
        thread.join(float(wait_seconds))
    alive = thread is not None and thread.is_alive()
    if not alive: media_thread = None
    return {'stopped': not alive, 'stopping': alive, 'last_recording': last_recording_path, 'last_events': last_event_path}

def recording_link():
    if not last_recording_path or not os.path.isfile(last_recording_path):
        return {'available': False, 'reason': 'no completed recording in this kernel'}
    relative = os.path.relpath(last_recording_path, os.getcwd())
    display(FileLink(relative, result_html_prefix='Download latest recording: '))
    if last_event_path and os.path.isfile(last_event_path):
        display(FileLink(os.path.relpath(last_event_path, os.getcwd()), result_html_prefix='Download synchronized events: '))
    return {'available': True, 'path': last_recording_path, 'events': last_event_path}

def _run_full_worker(rt, path, compact):
    global demo_thread, demo_thread_runtime
    status = 'SUCCESS'
    start = time.time()
    try:
        with open(path, 'a') as log_file:
            widget_stream = WidgetStream(output)
            display_stream = ImportantStream(widget_stream) if compact else widget_stream
            stdout_tee = Tee(log_file, display_stream)
            stderr_tee = Tee(log_file, WidgetStream(output, stderr=True))
            with routed_output(stdout_tee, stderr_tee):
                print('\n>>> run_full_background log={}'.format(path))
                try:
                    result = rt.run(cleanup=False)
                finally:
                    media_result = stop_media(wait_seconds=10.0)
                    print('[media] shutdown result={}'.format(media_result))
                    rt.stop_all()
                print('[result] {}'.format(result))
                if rt.stop_requested:
                    status = 'STOPPED'
                elif not result:
                    status = 'FAILED'
                print('[integration] {} elapsed={:.1f}s'.format(status, time.time() - start))
    except Exception as exc:
        status = 'FAILED'
        output.append_stderr('[error] background demo failed: {}\n'.format(exc))
        output.append_stderr(traceback.format_exc())
    finally:
        demo_thread = None
        demo_thread_runtime = None
        output.append_stdout('[integration] BACKGROUND RUN FINISHED status={} log={}\n'.format(status, path))
        scroll_log()

def run_full():
    global demo_thread, demo_thread_runtime
    if demo_thread is not None and demo_thread.is_alive():
        return {'started': False, 'reason': 'demo already running'}
    rt = current()
    if rt.state.value in ('DONE', 'FAILED'):
        raise RuntimeError('runtime is {}; click Reset Runtime, then Reload Config'.format(rt.state.value))
    path = default_log_path('integration_run_full')
    demo_thread_runtime = rt
    if live_stream.value or record_camera.value:
        start_media(start_camera=False)
    demo_thread = threading.Thread(target=_run_full_worker, args=(rt, path, bool(compact_log.value)))
    demo_thread.daemon = True
    demo_thread.start()
    return {'started': True, 'log': path}

def pause(): current().request_pause(); return show_state()
def resume(): current().resume(); return show_state()
def stop_base():
    rt = demo_thread_runtime or current()
    rt.request_pause()
    rt.services.base.stop()
    return {'paused': True, 'state': rt.state.value}
def stop_arm():
    rt = demo_thread_runtime or current()
    rt.request_pause()
    positions = rt.services.arm.stop_and_hold()
    return {'paused': True, 'state': rt.state.value, 'held_positions': positions}
def stop_camera():
    media_result = stop_media(wait_seconds=5.0)
    if not media_result['stopped']: return media_result
    current().release_camera()
    return show_state()
def stop_all():
    rt = demo_thread_runtime or current()
    rt.request_stop()
    media_result = stop_media(wait_seconds=0.0)
    worker_alive = demo_thread is not None and demo_thread.is_alive()
    if not worker_alive: rt.stop_all()
    return {'stop_requested': True, 'state': rt.state.value, 'cleanup_owner': 'background_worker' if worker_alive else 'caller', 'media': media_result}
def release_camera():
    media_result = stop_media(wait_seconds=5.0)
    if not media_result['stopped']: return media_result
    current().release_camera()
    return show_state()
def reset_runtime():
    global runtime
    if demo_thread is not None and demo_thread.is_alive():
        raise RuntimeError('demo is running; click STOP ALL and wait for BACKGROUND RUN FINISHED')
    media_result = stop_media(wait_seconds=5.0)
    if not media_result['stopped']: raise RuntimeError('media thread did not stop; runtime reset blocked')
    if runtime is not None: runtime.stop_all()
    runtime = None
    return True
def clear_log():
    output.clear_output(wait=False)
    return True

items = [('Reload Config', reload_config), ('Show State', show_state), ('Start Camera', start_camera), ('Load Models', load_models), ('Preflight', preflight), ('Step Once', step_once), ('Run Full Demo', run_full), ('Start Media', start_media), ('Stop Media', stop_media), ('Pause', pause), ('Resume', resume), ('STOP BASE', stop_base), ('STOP ARM', stop_arm), ('STOP CAMERA', stop_camera), ('STOP ALL', stop_all), ('Release Camera', release_camera), ('Reset Runtime', reset_runtime), ('Recording Link', recording_link), ('Clear Log', clear_log)]
buttons = []
for label, fn in items:
    button = widgets.Button(description=label, layout=widgets.Layout(width='150px'))
    if 'STOP' in label: button.button_style = 'danger'
    if label == 'Run Full Demo': button.button_style = 'success'
    if label == 'Clear Log':
        button.on_click(lambda _button: clear_log())
    elif label in ('Run Full Demo', 'Start Media', 'Stop Media', 'Pause', 'Resume', 'STOP BASE', 'STOP ARM', 'STOP CAMERA', 'STOP ALL', 'Release Camera'):
        button.on_click(immediate(fn))
    else:
        button.on_click(logged(fn))
    buttons.append(button)
display(widgets.VBox([widgets.HBox([reload_models, compact_log, camera_real, base_real, arm_real, exp2]), widgets.HBox([routine2, live_stream, record_camera, hud_enabled, media_fps, preview_fps]), widgets.HBox([static_detection, static_target, detection_fps]), widgets.HBox(buttons[0:4]), widgets.HBox(buttons[4:8]), widgets.HBox(buttons[8:12]), widgets.HBox(buttons[12:16]), widgets.HBox(buttons[16:19]), image, output]))
scroll_log()
print('[integration] ready; edit JSON, then Reload Config')
